# 2 - Paragraph classification model

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict

from sklearn.metrics import accuracy_score, classification_report, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

In [2]:
p_df = pd.read_parquet("legal_paragraphs_dataset.parquet")

In [3]:
p_df.head()

,document_id,paragraph_index,context,text,section,label,split
0,116061850,1,,Справа № 127/30543/23,introductory,0,train
1,116061850,2,Справа № 127/30543/23,Провадження № 1-кп/127/904/23,introductory,0,train
2,116061850,3,Справа № 127/30543/23\n\nПровадження № 1-кп/12...,В И Р О К,introductory,0,train
3,116061850,4,Провадження № 1-кп/127/904/23\n\nВ И Р О К,ІМЕНЕМ УКРАЇНИ,introductory,0,train
4,116061850,5,В И Р О К\n\nІМЕНЕМ УКРАЇНИ,28.12.2023 |місто Вінниця|,introductory,0,train


In [4]:
p_df.shape

(289367, 7)

In [5]:
columns = ["context", "text", "label"]
p_hf_dataset = DatasetDict({
    split: Dataset.from_pandas(
        p_df.loc[p_df["split"] == split, columns].reset_index(drop=True),
        preserve_index=False,
    )
    for split in ["train", "validation", "test"]
})

In [6]:
p_hf_dataset

DatasetDict({
    train: Dataset({
        features: ['context', 'text', 'label'],
        num_rows: 231493
    })
    validation: Dataset({
        features: ['context', 'text', 'label'],
        num_rows: 28929
    })
    test: Dataset({
        features: ['context', 'text', 'label'],
        num_rows: 28945
    })
})

## Loading encoder

In [7]:
encoder_path = "bge_model/bge-small-uk-law-v1_1/final/encoder"
tokenizer_path = "bge_model/bge-small-uk-law-v1_1/final/tokenizer"

In [8]:
LABELS_TO_IDS = {
    value: key for key, value in enumerate(p_df["section"].unique().tolist())
}

IDS_TO_LABELS = {
    key: value for key, value in enumerate(p_df["section"].unique().tolist())
}

In [9]:
LABELS_TO_IDS, IDS_TO_LABELS, len(LABELS_TO_IDS), len(IDS_TO_LABELS)

({'introductory': 0, 'reasoning': 1, 'operative': 2},
 {0: 'introductory', 1: 'reasoning', 2: 'operative'},
 3,
 3)

In [10]:
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
model = AutoModelForSequenceClassification.from_pretrained(
    encoder_path,
    num_labels=3,
    id2label=IDS_TO_LABELS,
    label2id=LABELS_TO_IDS,
)

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `1`.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bge_model/bge-small-uk-law-v1_1/final/encoder
Key                      | Status  | 
-------------------------+---------+-
bert.pooler.dense.bias   | MISSING | 
bert.pooler.dense.weight | MISSING | 
classifier.weight        | MISSING | 
classifier.bias          | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
def tokenize(batch):
    return tokenizer(
        batch["context"],
        batch["text"],
        max_length=512,
        padding=False,
    )

In [12]:
tokenized = p_hf_dataset.map(tokenize, batched=True, remove_columns=["context", "text"])

Map:   0%|          | 0/231493 [00:00<?, ? examples/s]

Map:   0%|          | 0/28929 [00:00<?, ? examples/s]

Map:   0%|          | 0/28945 [00:00<?, ? examples/s]

In [13]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Fine-Tuning model

In [14]:
def compute_metrics(prediction):
    logits, labels = prediction
    predicted_labels = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, predicted_labels),
        "macro_f1": f1_score(labels, predicted_labels, average="macro", zero_division=0),
    }

In [18]:
args = TrainingArguments(
    output_dir="model_v2",
    learning_rate=1e-5,
    weight_decay=0.01,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=25,
    seed=42,
)

In [19]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [20]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.231833,0.467689,0.900550,0.900997
2,0.229295,0.464594,0.903211,0.903318
3,0.239940,0.468358,0.904041,0.904070


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=43407, training_loss=0.23718455335046842, metrics={'train_runtime': 10112.945, 'train_samples_per_second': 68.672, 'train_steps_per_second': 4.292, 'total_flos': 3.2081796976133264e+16, 'train_loss': 0.23718455335046842, 'epoch': 3.0})

In [1]:
! cd /home/vegura/Projects/study/legal-data-doc-parts/src/legal_data_doc_parts && ../../.venv/bin/python /tmp/claude-1000/-home-vegura-Projects-study-legal-data-doc-parts/3ad2d9c5-fc16-4542-8498-941b94edc791/scratchpad/find_errors.py

test: 28945 paragraphs, 859 documents
Loading weights: 100%|█████████████████████| 201/201 [00:00<00:00, 3830.29it/s]
device=cuda  checkpoint=model_v2/checkpoint-43407
  0/28945
  5120/28945
  10240/28945
  15360/28945
  20480/28945
[W806 18:58:35.100111151 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 306184192 bytes (free: 49676288, total: 8220901376).
  25600/28945

accuracy = 0.9150
macro_f1 = 0.9109

              precision    recall  f1-score   support

introductory     0.9471    0.8209    0.8795      8643
   reasoning     0.9090    0.9574    0.9326     13728
   operative     0.8929    0.9503    0.9207      6574

    accuracy                         0.9150     28945
   macro avg     0.9164    0.9095    0.9109     28945
weighted avg     0.9168    0.9150    0.9140     28945

confusion matrix (rows=true, cols=pred), order intro/reasoning/operative
  introductory    7095    1058     490
  reasoning        326   13143     259
  o